# Ambiguity-code robustness

**Question answered.** Real clinical Sanger sequences of HIV pol routinely contain IUPAC
ambiguity codes (R, Y, K, M, S, W...) at positions where the quasispecies is mixed, typically
0.5-2% of positions. A 6-mer is destroyed if it contains even one ambiguous base, so this
compounds badly with fragmentation. This notebook (a) measures how much ambiguity is already
present in your dataset, (b) injects controlled ambiguity rates and re-evaluates every model
at every fragment length, and (c) produces the 2-D surface (ambiguity rate x fragment length)
the reviewer asks for.

**GPU needed.** No for the classical models (Linear SVM, Random Forest) which are the main
result here. Optionally add GPU if you also want the CNNs on this grid (much larger compute:
6 ambiguity rates x 4 lengths x N seeds x CNNs -- consider running CNNs only at 2-3 ambiguity
rates if GPU time is tight).

**Runtime.** Classical-only grid (6 ambiguity rates x 4 lengths x 3 seeds x 2 models = 144
fast fits): well under an hour, CPU only.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 2. Load data and measure the ambiguity ALREADY present

Your Methods section says 'bases other than A, C, G, or T were rewritten as N' but never reports how many there were. This cell reports that number directly from the raw sequences.

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
grp_all = df["cluster"].values

def ambiguity_rate(seq):
    non_acgt = sum(1 for c in seq if c not in "ACGT")
    return non_acgt / max(len(seq), 1)

rates = np.array([ambiguity_rate(s) for s in X_all])
print(f"Existing ambiguity rate in the cleaned dataset: mean={rates.mean()*100:.3f}%, "
      f"median={np.median(rates)*100:.3f}%, max={rates.max()*100:.3f}%")
print(f"Sequences with >0 ambiguous positions: {(rates>0).sum()} / {len(rates)} "
      f"({100*(rates>0).mean():.1f}%)")
pd.DataFrame({"ambiguity_rate": rates}).describe().to_csv(f"{OUT}/ambigbaseline_baseline_ambiguity_stats.csv")


### 3. Ambiguity injector (simulates mixed quasispecies positions with IUPAC codes)

In [ ]:

AMBIG = list("RYKMSWBDHVN")

def inject_ambiguity(seq, rate, rng):
    """Simulate quasispecies mixed positions. Real HIV clinical sequences carry these at
    0.5-2% of positions; degraded samples much higher."""
    if rate <= 0:
        return seq
    s = list(seq)
    n_to_change = int(len(s) * rate)
    if n_to_change == 0:
        return seq
    idx = rng.choice(len(s), size=n_to_change, replace=False)
    for i in idx:
        s[i] = rng.choice(AMBIG)
    return "".join(s)

def fragment(seq, length, rng):
    if length is None or len(seq) <= length:
        return seq
    start = rng.integers(0, len(seq) - length + 1)
    return seq[start:start+length]


### 4. Config

In [ ]:

AMBIG_RATES = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]
LENGTHS = [300, 500, 1000, None]   # None = full length
SEEDS = [0, 1, 2]
MODELS = {
    "linear_svm": lambda seed: LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000),
    "random_forest": lambda seed: RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                            random_state=seed, n_jobs=-1),
}


### 5. Main sweep: (ambiguity rate x fragment length x seed x model)

In [ ]:

def grouped_split(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    return trv, te

rows = []
for seed in SEEDS:
    tr, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yte = y_all[tr], y_all[te]
    rng = np.random.default_rng(seed)

    for L in LENGTHS:
        seq_tr_clean = [fragment(X_all[i], L, rng) for i in tr]
        # ambiguity is injected on TEST sequences only (simulates degraded clinical
        # input arriving at inference time; the model is trained on clean data, matching
        # how a deployed classifier would actually be trained once and used on real,
        # possibly-ambiguous samples)
        vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
        Xtr = vec.fit_transform(seq_tr_clean)

        for a_rate in AMBIG_RATES:
            seq_te = [inject_ambiguity(fragment(X_all[i], L, rng), a_rate, rng) for i in te]
            Xte = vec.transform(seq_te)

            for model_name, model_fn in MODELS.items():
                clf = model_fn(seed)
                clf.fit(Xtr, ytr)
                pred = clf.predict(Xte)
                rows.append(dict(
                    model=model_name, length=(L if L else "full"), ambiguity_rate=a_rate, seed=seed,
                    f1=f1_score(yte, pred, average="macro"),
                    bal_acc=balanced_accuracy_score(yte, pred)))
        pd.DataFrame(rows).to_csv(f"{OUT}/ambigbaseline_ambiguity_raw.csv", index=False)  # incremental save
        print(f"seed={seed} length={L} done")


### 6. Aggregate and plot the 2-D surface (ambiguity rate x fragment length)

In [ ]:

raw = pd.DataFrame(rows)
summ = raw.groupby(["model","length","ambiguity_rate"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)
summ.columns = [f"{a}_{b}" for a,b in summ.columns]
summ = summ.reset_index()
summ.to_csv(f"{OUT}/ambigbaseline_ambiguity_summary.csv", index=False)
print(summ)

length_order = [300, 500, 1000, "full"]
for model_name in raw["model"].unique():
    sub = summ[summ["model"] == model_name]
    pivot = sub.pivot(index="ambiguity_rate", columns="length", values="f1_mean")
    pivot = pivot[[c for c in length_order if c in pivot.columns]]
    fig, ax = plt.subplots(figsize=(6,4.5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels([f"{r*100:.1f}%" for r in pivot.index])
    ax.set_xlabel("fragment length (bp)"); ax.set_ylabel("injected ambiguity rate")
    ax.set_title(f"AmbigBaseline: macro-F1 surface — {model_name}")
    plt.colorbar(im, ax=ax, label="macro-F1")
    plt.tight_layout()
    plt.savefig(f"{OUT}/ambigbaseline_surface_{model_name}.png", dpi=200)
    plt.show()


### 7. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r ambigbaseline_ambiguity_outputs.zip ambigbaseline_ambiguity_raw.csv ambigbaseline_ambiguity_summary.csv ambigbaseline_baseline_ambiguity_stats.csv ambigbaseline_surface_*.png